# Nykaa Domain Support Agent — Task 14: Expose Order Lookup via MCP

## Objective (from the capstone brief)
Expose the existing Task 6 function `check_order_status(record_id: str) -> dict` as a real Model Context Protocol (MCP) tool.

The acceptance criteria are:
1. Install the package **`fastmcp`** (not `fast-mcp`).
2. Wrap `check_order_status` with a clear, proper docstring.
3. Launch a private MCP server locally using HTTP transport.
4. Use the MCP endpoint at **`http://127.0.0.1:8001/mcp`**. FastMCP's HTTP transport uses `/mcp` by default; the bare host and port are not the MCP endpoint.
5. Write a **separate client script** (a different file/process from the LangGraph agent).
6. Successfully call the MCP tool for at least **two different order IDs** and print the standardized MCP responses.

This task reuses your existing Task 6 lookup function. It must not create a second, differently calculated escalation formula.

**Run this notebook from the project folder containing your existing `app.py`, order dataset, and other project files.**

## 1. Check the project files and locate the Task 6 lookup function

In [ ]:
from pathlib import Path
import importlib
import inspect
import pandas as pd

PROJECT_DIR = Path.cwd()
APP_FILE = PROJECT_DIR / "app.py"

print("Project directory:", PROJECT_DIR.resolve())
print("app.py exists:", APP_FILE.exists())

if not APP_FILE.exists():
    raise FileNotFoundError(
        "app.py was not found. Run this notebook from the same project folder "
        "where your Task 11 FastAPI app.py is stored."
    )

import app as project_app

if not hasattr(project_app, "check_order_status"):
    raise AttributeError(
        "Your app.py does not expose check_order_status. "
        "Import the exact Task 6 function into app.py, or adjust the import "
        "in the MCP server cell to the Python module where you defined it."
    )

check_order_status = project_app.check_order_status

print("PASS: Existing Task 6 check_order_status function found.")
print("Function signature:", inspect.signature(check_order_status))
print("Function module:", check_order_status.__module__)


## 2. Install FastMCP

The required package name is `fastmcp` (no hyphen).

In [ ]:
%pip install -U fastmcp

## 3. Test the existing order lookup before exposing it

This confirms that the MCP layer will call the same Task 6 function and return its existing result structure.

In [ ]:
# Test two different IDs already used in the synthetic project.
TEST_ORDER_IDS = ["ORD1001", "ORD1002"]

for order_id in TEST_ORDER_IDS:
    result = check_order_status(order_id)
    print(f"\nORDER ID: {order_id}")
    print(result)
    assert isinstance(result, dict), "check_order_status must return a dictionary."

assert TEST_ORDER_IDS[0] != TEST_ORDER_IDS[1]
print("\nPASS: Existing order lookup responds for two different record IDs.")


## 4. Create the private MCP server (`mcp_server.py`)

The MCP server imports the existing `check_order_status` function from `app.py` and registers it as a tool. The tool docstring describes its purpose, input, and output.

The server uses port **8001** so it can run alongside your existing FastAPI application on port 8000.

In [ ]:
%%writefile mcp_server.py
from fastmcp import FastMCP
from app import check_order_status as task6_check_order_status

mcp = FastMCP("Nykaa Order Lookup MCP Server")

@mcp.tool
def check_order_status(record_id: str) -> dict:
    """Look up a Nykaa synthetic order and return its status and escalation assessment.

    Args:
        record_id: The unique order identifier, for example "ORD1001".

    Returns:
        A dictionary containing whether the order was found and, when available,
        its status, order value, escalation score, escalation threshold, and
        whether human escalation is recommended.
    """
    if not isinstance(record_id, str) or not record_id.strip():
        return {
            "found": False,
            "error": "Please provide a valid order ID."
        }

    # Delegate to the original Task 6 implementation. Do not duplicate its rules.
    result = task6_check_order_status(record_id.strip())
    return result


if __name__ == "__main__":
    # FastMCP HTTP transport mounts the MCP endpoint at /mcp by default.
    mcp.run(
        transport="http",
        host="127.0.0.1",
        port=8001
    )


## 5. Create the separate MCP client (`mcp_client.py`)

This client is a separate Python file and process from the LangGraph agent. It connects to the MCP server's `/mcp` endpoint, discovers the registered tool, calls it for two different order IDs, and prints the returned MCP result content.

In [ ]:
%%writefile mcp_client.py
import asyncio
import json
from fastmcp import Client

MCP_SERVER_URL = "http://127.0.0.1:8001/mcp"

async def main():
    # Client connects to the MCP HTTP endpoint, not the bare host:port root.
    async with Client(MCP_SERVER_URL) as client:
        tools = await client.list_tools()
        tool_names = [tool.name for tool in tools]

        print("TOOLS DISCOVERED:", tool_names)
        assert "check_order_status" in tool_names, (
            "The MCP server did not expose check_order_status."
        )

        # Two distinct order IDs are required by the capstone brief.
        for record_id in ("ORD1001", "ORD1002"):
            print("\\n" + "=" * 65)
            print("MCP TOOL CALL:", record_id)

            result = await client.call_tool(
                "check_order_status",
                {"record_id": record_id}
            )

            # Print the standardized MCP result object as JSON when possible.
            if hasattr(result, "model_dump"):
                output = result.model_dump(mode="json")
            elif hasattr(result, "dict"):
                output = result.dict()
            else:
                output = str(result)

            print(json.dumps(output, indent=2, ensure_ascii=False, default=str))

            # FastMCP CallToolResult commonly exposes structured_content.
            structured = getattr(result, "structured_content", None)
            if structured is not None:
                print("STRUCTURED CONTENT:")
                print(json.dumps(structured, indent=2, ensure_ascii=False, default=str))

            # If tool execution returned an error, fail the client test.
            if getattr(result, "is_error", False):
                raise RuntimeError(f"MCP tool call failed for {record_id}")

    print("\\nPASS: MCP client successfully called check_order_status for two IDs.")

if __name__ == "__main__":
    asyncio.run(main())


## 6. Launch the MCP server in a separate VS Code terminal

Keep your existing FastAPI server on port 8000 if you need it. Open a **second** VS Code terminal, make sure it is in the project directory, and run:

```powershell
python mcp_server.py
```

The MCP server should start on:

```text
http://127.0.0.1:8001/mcp
```

Leave this terminal running. Do not try to open `/mcp` as an ordinary browser page; it is an MCP transport endpoint, not a human-facing Swagger page.

If the server reports that port 8001 is already in use, stop the other process using that port or change the port in both `mcp_server.py` and `mcp_client.py`.

## 7. Run the separate MCP client

With the MCP server still running in the other terminal, open a **third terminal** in the same project directory and execute:

```powershell
python mcp_client.py
```

Alternatively, run the following cell in this notebook while the server is running.

In [ ]:
import subprocess
import sys

client_script = PROJECT_DIR / "mcp_client.py"
assert client_script.exists(), "mcp_client.py was not created."

completed = subprocess.run(
    [sys.executable, str(client_script)],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    timeout=60
)

print(completed.stdout)
if completed.stderr:
    print("STDERR:")
    print(completed.stderr)

assert completed.returncode == 0, (
    "MCP client did not complete successfully. Check that mcp_server.py "
    "is still running in the separate terminal."
)

assert "PASS: MCP client successfully called check_order_status for two IDs." in completed.stdout
print("PASS: Separate MCP client process completed.")


## 8. Task 14 acceptance checklist

Task 14 is complete when you can show all of the following:
- `fastmcp` is installed in the same Python environment.
- `mcp_server.py` starts a local HTTP MCP server.
- The MCP tool is named `check_order_status` and has a proper docstring.
- The client connects to `http://127.0.0.1:8001/mcp`.
- The separate `mcp_client.py` process discovers and calls the tool.
- Calls for two different record IDs return the standardized MCP response without errors.

### Files created
- `mcp_server.py` — MCP server and tool wrapper.
- `mcp_client.py` — separate MCP client.

### Important integration note
The server imports `check_order_status` from your existing `app.py`. If that function currently exists only inside a notebook and not in `app.py`, move/reuse the exact Task 6 implementation in a Python module imported by `app.py`; do not create a new scoring formula for MCP.

The next task after this one is **Task 15 — SQLite-based LangGraph checkpointing**. It is separate from MCP and will use `checkpoints.sqlite` keyed by a thread ID.